In [0]:
%sql
-- Task 1: Channel performance

with Revenue_Summary as (
     select o.channel, 
          round (sum (case when 
          order_date <= '2024-06-30'
            then l.line_revenue
          else 0 end),2) as Revenue24, 
     round (sum ( case when 
        order_date > '2024-06-30'
           then l.line_revenue
        else 0 end),2) as Revenue25
          
from sizwe_sales_orders as o
inner join sizwe_sales_lines as l
on o.order_id = l.order_id
group by o.channel )

select channel, Revenue24, Revenue25, 
round(((Revenue25 - Revenue24)/nullif (Revenue24,0))*100,1) as Growth_Rate 
from Revenue_Summary 


channel,Revenue24,Revenue25,Growth_Rate
Wholesale,3.0816601232E8,3.3542632502E8,8.8
Subscription,1.3283398515E8,1.4457369833E8,8.8


In [0]:
%sql 
-- Task 2: Where the profit actually comes from?

with Revenue_Summary as (
    select o.channel, 
        round (sum (case when 
        order_date <= '2024-06-30'
           then l.line_revenue
        else 0 end),2) as Revenue24,

round (sum ( case when 
    order_date > '2024-06-30'
       then l.line_revenue
    else 0 end),2) as Revenue25
      
from sizwe_sales_orders as o
inner join sizwe_sales_lines as l
on o.order_id = l.order_id
group by o.channel )

select channel, Revenue24, Revenue25, 
round(((Revenue25 - Revenue24)/nullif (Revenue24,0))*100,1) as Growth_Rate 
from Revenue_Summary;

with Margins as (
    select p.product_line as Product_Line,
    round(sum(line_revenue),2) as Total_Revenue,
    round(sum(cogs),2) as Total_Cost, 
    round(sum(line_revenue) - sum(cogs),2) as Gross_Profit, 
    round(((sum(line_revenue) - sum(cogs)) / nullif(sum(line_revenue),0)*100),2) as Gross_Mar_Per
from sizwe_sales_lines sl
inner join sizwe_products p
on sl.sku = p.sku
group by p.product_line
)

select Product_Line, Total_Revenue, Total_Cost,Gross_Profit, Gross_Mar_Per
from Margins
order by Gross_Profit desc

--- The House Blend line earns the highest absolute gross profit. However, it is not the same line that achieves the highest efficiency, as the highest gross margin percentage belongs to the Single Origin line.

Product_Line,Total_Revenue,Total_Cost,Gross_Profit,Gross_Mar_Per
House Blend,2.795309489E8,1.723822023E8,1.071487466E8,38.33
Single Origin,1.7011261961E8,9.479981874E7,7.531280087E7,44.27
Bulk Commercial,2.7183883101E8,2.1773595153E8,5.410287948E7,19.9
Capsules,1.0887507622E8,6.651657155E7,4.235850467E7,38.91
Decaf,9.064254508E7,5.816754701E7,3.247499807E7,35.83


In [0]:
%sql
-- Task 3 : The collections problem

with customer_channels as (
    select distinct customer_id, channel
    from sizwe_ar_ledger
    where channel is not null
    )

select
    c.channel,
    count(*) as Total_Invoices,
    count(case when i.amount_settled > 0 then 1 end) as Settled_Invoices,
    count(case when i.amount_settled = 0 then 1 end) as Open_Invoices,
    round(avg(case
        when i.settlement_date is not null
        then datediff(i.settlement_date, i.invoice_date)
        end), 2 ) as Avg_Days_to_Settle,
    round(sum(i.invoice_amount - i.amount_settled), 2) as Total_Outstanding_Value
from sizwe_ar_ledger as i
inner join customer_channels as c
on i.customer_id = c.customer_id
where i.invoice_no is not null
group by c.channel
order by Total_Outstanding_Value desc 

--- Wholesale is the main collections concern, with 3,454 open invoices worth R87,848,227.20 and an average settlement time of 68.68 days. In comparison, Subscription has only 44 open invoices worth R751,178.89 and an average settlement time of 3.85 days. Nomsa should investigate the Wholesale channel first because it has substantially more outstanding debt and slower collections.


channel,Total_Invoices,Settled_Invoices,Open_Invoices,Avg_Days_to_Settle,Total_Outstanding_Value
Wholesale,28944,25490,3454,68.68,8.78482272E7
Subscription,19200,19156,44,3.85,751178.89


In [0]:
%sql
-- Task 4 : The ten accounts that matter most

with Revenue_Summary as(
    select
    c.customer_id, 
    c.trading_name,
    c.segment,
    c.province,
    round(sum(case when o.order_date between '2024-07-01' and '2025-06-30' 
        then l.line_revenue 
        else 0 end), 2) as Revenue25  
    from sizwe_sales_orders as o
    inner join sizwe_sales_lines as l on o.order_id = l.order_id
    inner join sizwe_customers as c on o.customer_id = c.customer_id 
    group by c.customer_id, c.trading_name, c.segment, c.province
)
select 
     r.trading_name, 
     r.segment, 
     r.province, 
     max(sr.rep_name) as Sales_Rep,
     r.Revenue25 
from Revenue_Summary as r
left join sizwe_rep_visits as v on r.customer_id = v.customer_id
left join sizwe_reps as sr on v.rep_id = sr.rep_id
group by r.trading_name, r.segment, r.province, r.Revenue25
order by r.Revenue25 desc
limit 10



trading_name,segment,province,Sales_Rep,Revenue25
Halvane Retail Group,Grocery Chain,Western Cape,Kagiso Mkhize,3.839999989E7
Highveld Deli 85,Restaurant Group,Free State,Karabo Mokoena,3380200.33
Table Bay Foods 280,Independent Cafe,KwaZulu-Natal,Karabo van Wyk,3333678.0
Umhlanga Fresh 108,Restaurant Group,Gauteng,Sipho Steyn,3246754.35
Kalahari Kitchen 42,Independent Cafe,Gauteng,Chantal Ismail,2320923.25
Ridgeway Foods 308,Independent Cafe,KwaZulu-Natal,Chantal Adams,2320555.37
Boland Bakery 91,Independent Cafe,Free State,Karabo Zwane,2280097.99
The Daily Grind 219,Hotel Group,Western Cape,Nomsa Botha,2134607.32
Midrand Market 293,Independent Cafe,KwaZulu-Natal,Chantal Ismail,2024456.98
Bloom Coffee House 275,Hotel Group,Gauteng,Karabo Zwane,1934391.39


In [0]:
%sql 
-- Task 5 : A commercial gap

with Customer_Revenue as (
    select 
        c.customer_id,
        c.trading_name,
        c.channel,      
        c.segment,
        c.province,
        round(sum(l.line_revenue), 2) as Total_Revenue
    from sizwe_customers as c
    inner join sizwe_sales_orders as o on c.customer_id = o.customer_id
    inner join sizwe_sales_lines as l on o.order_id = l.order_id
    group by c.customer_id, c.trading_name, c.channel, c.segment, c.province
)

select cr.trading_name, cr.channel, cr.segment, cr.province, cr.Total_Revenue
from customer_revenue as cr
left join sizwe_trade_promotions as p 
on cr.customer_id = p.customer_id
where cr.channel = "Wholesale"  
and p.customer_id is null
order by cr.total_revenue desc
limit 15 

--- I recommend proactively reaching out to the top three wholesale customers to offer them tailored trade promotions or volume-based discounts to reward their loyalty, secure their accounts, and prevent them from churning to competitors



trading_name,channel,segment,province,Total_Revenue
Umhlanga Fresh 108,Wholesale,Restaurant Group,Gauteng,6224772.37
Ridgeway Foods 308,Wholesale,Independent Cafe,KwaZulu-Natal,4448744.72
Kalahari Kitchen 42,Wholesale,Independent Cafe,Gauteng,4447225.01
Midrand Market 293,Wholesale,Independent Cafe,KwaZulu-Natal,3887087.18
Bloom Coffee House 275,Wholesale,Hotel Group,Gauteng,3719361.46
Corner Perk 34,Wholesale,Independent Cafe,Limpopo,3431735.39
Highveld Deli 28,Wholesale,Independent Cafe,Gauteng,3421285.02
Bloom Coffee House 306,Wholesale,Restaurant Group,Gauteng,3397003.31
Kalahari Kitchen 348,Wholesale,Independent Cafe,KwaZulu-Natal,3270408.02
Zebra Cafe 201,Wholesale,Grocery Chain,Free State,3169175.74


In [0]:
%sql 
--Task 6 : Are the sales representatives hitting target?

with Rep_Revenue_Summary as (
select 
   r.rep_name,
   r.territory,
   r.target_monthly,
   round(sum( case 
        when o.order_date between '2024-07-01' and '2025-06-30'
        then l.line_revenue
        else 0 end),2) as total_revenue 
from sizwe_reps as r
left join sizwe_sales_orders as o
on o.rep_id = r.rep_id
left join sizwe_sales_lines as l
on o.order_id = l.order_id
group by r.rep_name, r.territory, r.target_monthly 
)

select rs.rep_name,
       rs.territory,
       rs.target_monthly,
       round (rs.total_revenue / 12,2) as avg_mon_rev,
       round ((avg_mon_rev / target_monthly)*100,2) 
       as avg_mon_rev_per
from Rep_Revenue_Summary as rs
order by avg_mon_rev_per desc



rep_name,territory,target_monthly,avg_mon_rev,avg_mon_rev_per
Karabo van Wyk,Gauteng South,900000,2252880.96,250.32
Kagiso Mkhize,Gauteng North,2000000,4996325.67,249.82
Johan Maluleke,KwaZulu-Natal,900000,2184885.21,242.77
Anele Naidoo,Western Cape,900000,2011837.24,223.54
Anele Zwane,Northern Region,900000,1846514.09,205.17
Karabo Mokoena,Gauteng South,1200000,2365324.98,197.11
Anele Ismail,Eastern Cape,1200000,2299147.69,191.6
Chantal Ismail,Central Region,1200000,2285301.19,190.44
Pieter Pillay,Central Region,1200000,2068510.95,172.38
Nomsa Botha,Gauteng South,1500000,2377843.54,158.52


In [0]:
%sql 
-- Task 7 : The shape of the year

with Monthly_Revenue as (
    select 
    date_trunc ('month', o.order_date)
    as Revenue_Month,  
    round(sum (l.line_revenue),2) 
    as Month_Revenue

    from sizwe_sales_orders as o
    inner join sizwe_sales_lines as l
    on o.order_id = l.order_id
    where o.order_date between '2023-07-01' and '2025-06-30'
    group by date_trunc ('month', o.order_date)
),

Revenue_Analysis as (
    select   
       Revenue_Month, 
       Month_Revenue,
       lag(Month_Revenue,1) 
       over(order by Revenue_Month)as Previous_Mon_Revenue,
       avg(Month_Revenue)
       over(order by Revenue_Month
       rows between 2 preceding and current row)
       as Rolling_3Month_Avg   
       from Monthly_Revenue 
)

select 
       Revenue_Month, 
       Month_Revenue,
       Previous_Mon_Revenue,
       round(100*(Month_Revenue - Previous_Mon_Revenue)/nullif(Previous_Mon_Revenue,0),2) as Percentage_Change,
       round(Rolling_3Month_Avg,2) as Rolling_3Month_Avg
       from Revenue_Analysis
       order by Revenue_Month;


--Sizwe's revenue shows a seasonal pattern, with higher revenue generally occurring between July and September and lower revenue during December and January, suggesting a possible slowdown around the festive season and start of the new year. December 2023 was Sizwe's quietest month, recording monthly revenue of R29,872,699.78.

Revenue_Month,Month_Revenue,Previous_Mon_Revenue,Percentage_Change,Rolling_3Month_Avg
2023-07-01T00:00:00.000Z,4.105147175E7,null,null,4.105147175E7
2023-08-01T00:00:00.000Z,3.960774434E7,4.105147175E7,-3.52,4.032960805E7
2023-09-01T00:00:00.000Z,3.680777906E7,3.960774434E7,-7.07,3.915566505E7
2023-10-01T00:00:00.000Z,3.578480569E7,3.680777906E7,-2.78,3.74001097E7
2023-11-01T00:00:00.000Z,3.437016262E7,3.578480569E7,-3.95,3.565424912E7
2023-12-01T00:00:00.000Z,2.987269978E7,3.437016262E7,-13.09,3.334255603E7
2024-01-01T00:00:00.000Z,3.125562108E7,2.987269978E7,4.63,3.183282783E7
2024-02-01T00:00:00.000Z,3.515602563E7,3.125562108E7,12.48,3.209478216E7
2024-03-01T00:00:00.000Z,3.728749758E7,3.515602563E7,6.06,3.456638143E7
2024-04-01T00:00:00.000Z,3.795072875E7,3.728749758E7,1.78,3.679808399E7


In [0]:
%sql 
-- Task 8 : Why has the margin moved?

with Product_Line_Revenue as (
    select
        case
            when o.order_date >= '2023-07-01'
            and o.order_date < '2024-07-01'
                then 'FY2024'
            when o.order_date >= '2024-07-01'
            and o.order_date < '2025-07-01'
                then 'FY2025'
        end as Financial_Year,

        p.product_line,
        sum(l.line_revenue) as Revenue,
        sum(l.cogs) as COGS

    from sizwe_sales_orders as o
    inner join sizwe_sales_lines as l
        on o.order_id = l.order_id
    inner join sizwe_products as p
        on l.sku = p.sku

    where o.order_date >= '2023-07-01'
    and o.order_date < '2025-07-01'

    group by
        case
            when o.order_date >= '2023-07-01'
            and o.order_date < '2024-07-01'
                then 'FY2024'
            when o.order_date >= '2024-07-01'
            and o.order_date < '2025-07-01'
                then 'FY2025'
        end,
        p.product_line
),

Product_Line_Analysis as (
    select
        Financial_Year,
        product_line,
        Revenue,
        COGS,

        100 * (Revenue - COGS)
            / NULLIF(Revenue, 0) as Gross_Margin_Pct,

        100 * Revenue
            / sum(Revenue) over (
                partition by Financial_Year
            ) as Revenue_Share_Pct

    from Product_Line_Revenue
)

select
    Financial_Year,
    product_line,
    round(Revenue, 2) as Revenue,
    round(Gross_Margin_Pct, 2) as Gross_Margin_Pct,
    round(Revenue_Share_Pct, 2) as Revenue_Share_Pct

from Product_Line_Analysis
order by
    Financial_Year,
    product_line

--The gross margin declined from 34.7% in FY2024 to 33.0% in FY2025 mainly because of a shift in the revenue mix towards Bulk Commercial, which has a much lower gross margin than the other product lines. Bulk Commercial's revenue share increased from 25.02% to 33.65%, while its margin remained around 19.9%; meanwhile, House Blend's share fell from 32.14% to 28.71% and Single Origin's from 19.45% to 17.57%, despite both maintaining margins above 38%. This indicates that the decline was driven primarily by selling a larger proportion of lower-margin products rather than a major deterioration in individual product-line margins.

Financial_Year,product_line,Revenue,Gross_Margin_Pct,Revenue_Share_Pct
FY2024,Bulk Commercial,1.1033858135E8,19.89,25.02
FY2024,Capsules,5.68223495E7,39.01,12.88
FY2024,Decaf,4.630047747E7,35.85,10.5
FY2024,House Blend,1.4174647148E8,38.32,32.14
FY2024,Single Origin,8.579211767E7,44.24,19.45
FY2025,Bulk Commercial,1.6150024966E8,19.91,33.65
FY2025,Capsules,5.205272672E7,38.79,10.84
FY2025,Decaf,4.434206761E7,35.81,9.24
FY2025,House Blend,1.3778447742E8,38.34,28.71
FY2025,Single Origin,8.432050194E7,44.31,17.57
